<a href="https://colab.research.google.com/github/diarraaaa/Funcode/blob/main/Book_recommadation_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
books=pd.read_csv('books_cleaned.csv',on_bad_lines='skip', encoding='latin1', engine='python')
#on va creer un champ qui va faciliter le calcul du TF-IDF
books['content']=(
    books['title'].fillna('') + ' ' +
    books['author'].fillna('') + ' ' +
    books['description'].fillna('') + ' ' +
    books['main_genre'].fillna('') + ' ' +
    books['main_genre'].fillna('') + ' ' +
    books['second_genre'].fillna('') + ' ' +
    books['series'].fillna('') + ' ' +
    books['publisher'].fillna('')
)
books.to_csv('books_with_content.csv', index=False)
#on va initialiser le TF-IDF vectorizer qui va nous permettre de transformer le champ 'content' en une matrice de caractéristiques numériques en ignorant les mots vides en anglais comme "the", "is", "in", etc.
convector=TfidfVectorizer(stop_words='english')
#on va appliquer le TF-IDF vectorizer sur le champ 'content' pour obtenir une matrice de caractéristiques numériques qui représente l'importance de chaque mot dans chaque livre et dans l'ensemble du corpus de livres
matrix=convector.fit_transform(books['content'])
#on va creer la fonction de recommendation de 5 livres en fonction de son titre
def recommend_books(titre,n=5):
    #on va d'abord verifier si le titre existe dans notre dataset
    exits=books[books['title'].str.contains(titre, case=False, na=False)]
    if exits.empty:
        print(f"The book '{titre}' do not exist in the dataset.")
        return
    #on va recuperer l'index du livre correspondant au titre
    indx=exits.index[0]
    #on va calculer la similarite cosinus de ce livre avec les autres livres
    cosine_sim=cosine_similarity(matrix[indx], matrix).flatten()
    #on va transformer le resulat en liste de tuples des cosinus
    similarity=list(enumerate(cosine_sim))
    #on trie les cosinus par ordre decroissant
    similarity=sorted(similarity,key=lambda x:x[1],reverse=True )
    #on recupere les n premiers qui correspondront aux livres qui seront recommandes
    recommended=similarity[1:n+1]
    print(f'If you liked {titre},you will like:')
    #on affiche les livres recommendes par le modele
    for couple in recommended:
      print(f'{books['title'][couple[0]]} by {books['author'][couple[0]]}')
recommend_books('harry potter',9)

If you liked harry potter,you will like:
The Hogwarts Library by J.K. Rowling
Harry Potter: A History of Magic by British Library, J.K. Rowling, Julian Harrison, Julia Eccleshare, Roger Highfield, Anna Pavord, Lucy Mangan, Tim Peake, Owen  Davies, Richard  Coles, Steve Backshall, Steve Kloves
James Potter and the Curse of the Gatekeeper by G. Norman Lippert
Selections from Harry Potter and the Order of the Phoenix: Piano Solos by John   Williams, Nicholas Hooper
The Leopard Prince by Elizabeth Hoyt
Very Good Lives: The Fringe Benefits of Failure and the Importance of Imagination by J.K. Rowling, Joel Holland
The Young Wizards by Diane Duane
Flyaway by Lucy Christopher
Trunk Music by Michael Connelly


# 1. TF-IDF (Term Frequency - Inverse Document Frequency)

Le TF-IDF est une technique qui transforme du **texte en nombres**. Il attribue un score à chaque mot selon deux critères combinés :

- **TF (Term Frequency)** : mesure l'importance d'un mot *dans un livre précis*. Plus un mot apparaît souvent dans ce livre, plus son TF est élevé.
- **IDF (Inverse Document Frequency)** : mesure la rareté d'un mot *dans tout le dataset*. Plus un mot est rare dans l'ensemble des livres, plus son IDF est élevé.

Le score final est : **TF-IDF = TF × IDF**

---
# 3. La similarité cosinus

La similarité cosinus compare deux vecteurs TF-IDF en mesurant l'**angle** entre eux. Plus l'angle est petit, plus les livres se ressemblent.

- Score **proche de 1** → les vecteurs pointent dans la même direction → livres très similaires
- Score **proche de 0** → les vecteurs sont perpendiculaires → livres très différents

Le résultat est une **matrice de similarité** (11 000 × 11 000) où chaque cellule représente le score entre deux livres.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
books=pd.read_csv('books_cleaned.csv',on_bad_lines='skip', encoding='latin1', engine='python')
#on va creer un champ qui va faciliter le calcul du TF-IDF
books['content']=(
    books['title'].fillna('') + ' ' +
    books['author'].fillna('') + ' ' +
    books['description'].fillna('') + ' ' +
    books['main_genre'].fillna('') + ' ' +
    books['main_genre'].fillna('') + ' ' +
    books['second_genre'].fillna('') + ' ' +
    books['series'].fillna('') + ' ' +
    books['publisher'].fillna('')
)
books.to_csv('books_with_content.csv', index=False)
#on va initialiser le TF-IDF vectorizer qui va nous permettre de transformer le champ 'content' en une matrice de caractéristiques numériques en ignorant les mots vides en anglais comme "the", "is", "in", etc.
convector=TfidfVectorizer(stop_words='english')
#on va appliquer le TF-IDF vectorizer sur le champ 'content' pour obtenir une matrice de caractéristiques numériques qui représente l'importance de chaque mot dans chaque livre et dans l'ensemble du corpus de livres
matrix=convector.fit_transform(books['content'])
#on va creer la fonction de recommendation des livres en fonction des interactions de l'utilisateur avec les autres livres du dataset
def recommand_interactions(interactions,n=10):
  #on creer une liste de 0 pour stocker toutes les notes
  score=np.zeros(len(books))
  #ensuite on parcourt la liste des interactions pour assigner les nouvelles notes de chaque livre
  for index,note in interactions.items():
    realindex=books[books['id']==index].index[0]
    similarity=cosine_similarity(matrix[realindex],matrix).flatten()
    score+=similarity*note
    #on va ensuite diminuer la note des livres deja notes pour ne pas les proposer
    score[realindex]=-10000
  #on va ensuite trier les scores par ordre decroissant
  score=list(enumerate(score))
  score=sorted(score,key=lambda x:x[1],reverse=True)
  #on va recuperer les n premiers qui correspondent a nos livres que nous allons recommander
  recommended=score[:n]
  print('Here are the books you might like:')
  for couple in recommended:
    print(f'{books['title'][couple[0]]} by {books['author'][couple[0]]}')
interactions={
    22037424:1.0, #signifie qu'il a aime le livre Harry Potter
    12926767:1.0, #signifie qu'il a aime le livre Romeo et Juliet
    129956:-1.0,#signifie qu'il n'a pas aime le livre Madame Bovary
}
recommand_interactions(interactions,15)



In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
books=pd.read_csv('books_cleaned.csv',on_bad_lines='skip', encoding='latin1', engine='python')
books['content']=(
    books['title'].fillna('') + ' ' +
    books['author'].fillna('') + ' ' +
    books['description'].fillna('') + ' ' +
    books['main_genre'].fillna('') + ' ' +
    books['second_genre'].fillna('') + ' ' +
    books['series'].fillna('') + ' ' +
    books['series'].fillna('') + ' ' +
    books['publisher'].fillna('')
)
books.to_csv('books_with_content.csv', index=False)
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')


In [ ]:
vec_matrix=np.load('vec_matrix.npy')
np.save('vec_matrix.npy',vec_matrix)
def recommand_interactions_word2vec(interactions,n=10):
  #on creer une liste de 0 pour stocker toutes les notes
  score=np.zeros(len(books))
  #ensuite on parcourt la liste des interactions pour assigner les nouvelles notes de chaque livre
  for index,note in interactions.items():
    realindex=books[books['id']==index]
    if realindex.empty:
      continue
    realindex=realindex.index[0]
    similarity=cosine_similarity(vec_matrix[realindex].reshape(1,-1),vec_matrix).flatten()
    score+=similarity*note
    #on va ensuite diminuer la note des livres deja notes pour ne pas les proposer
    score[realindex]=-10000
  #on va ensuite trier les scores par ordre decroissant
  score=list(enumerate(score))
  score=sorted(score,key=lambda x:x[1],reverse=True)
  #on va recuperer les n premiers qui correspondent a nos livres que nous allons recommander
  recommended=score[:n]
  print('Here are the books you might like:')
  resultat=[]
  for couple in recommended:
    resultat.append((books['id'][couple[0]],books['title'][couple[0]]))
  # Renaming 'id' to 'book_id' for consistency
  resultat=pd.DataFrame(resultat,columns=['book_id','title'])
  return resultat
interactions={
    2002:1.0, #signifie qu'il a aime le livre Harry Potter
    139421:1.0, #signifie qu'il a aime le livre The eternal Husband
}
print(recommand_interactions_word2vec(interactions,15))

Here are the books you might like:
     book_id                                              title
0   16177185                               The Hogwarts Library
1       8154                                 King, Queen, Knave
2    7816410                                   Immanuel's Veins
3     210190                                         The Double
4   19055104  Ã Â¦Â®Ã Â¦Â¾Ã Â¦Â²Ã Â¦Â¾Ã Â¦ÂÃ Â¦Â¾Ã Â¦ÂÃ Â¦...
5    1688292  A Familiar Dragon: Fanuilh / Wizard's Heir / B...
6       3532                                         First Love
7      11662                                            Couples
8      48203                              The Professor's House
9      31785                                  The Will to Power
10   6691280                                 Mr. Darcy, Vampyre
11  45167104                                          Angskandh
12  35613533                   Harry Potter: A History of Magic
13    407154                   What Men Live by and Other Tales
14  1

# Collaborative Filtering — Approche User-Based

## Principe

"Des utilisateurs qui ont les mêmes goûts dans le passé auront probablement les mêmes goûts dans le futur."

On ne se base pas sur le contenu des livres, mais sur le **comportement des utilisateurs**.

---

## Les étapes

**1. Construire la matrice utilisateur × livre**

On transforme les interactions (likes, notes) en une matrice où chaque ligne = un utilisateur et chaque colonne = un livre.

```
         HP     Dune   1984   Narnia
User 0  [ 1.0    0.0   -1.0    0.6  ]
User 1  [ 1.0    0.6    0.0    0.4  ]
User 2 [ 0.0    1.0    0.8    0.0  ]
```

**2. Calculer la similarité cosinus entre utilisateurs**

On compare les lignes de la matrice — plus deux utilisateurs ont noté les mêmes livres de façon similaire, plus leur score est proche de 1.

**3. Trouver les k utilisateurs les plus similaires**

On trie les scores et on sélectionne les k voisins les plus proches de l'utilisateur cible.

**4. Recommander**

On récupère les livres aimés par ces utilisateurs similaires que l'utilisateur cible n'a pas encore lus. On pondère chaque livre par la similarité de l'utilisateur qui l'a noté pour obtenir un score final.

---

## Limites

- **Cold start** — un nouvel utilisateur sans interactions ne peut pas recevoir de recommandations
- **Données sparse** — avec peu d'interactions, les similarités sont peu fiables
- **Scalabilité** — calculer la similarité entre des millions d'utilisateurs est coûteux

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
# Connexion à la base de données SQLite.
conn = sqlite3.connect('interactions.db')
cursor = conn.cursor()
# Cette table stocke les interactions des utilisateurs avec les livres (qui a noté quel livre et quelle note).
cursor.execute('''
    CREATE TABLE IF NOT EXISTS interactions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER,
        book_id INTEGER,
        note REAL,
        Unique(book_id,user_id)
    )
''')
# Cette table stocke les informations de base des livres qui ont été notés par au moins un utilisateur.
cursor.execute('''
    CREATE TABLE IF NOT EXISTS books_rated (
        book_id INTEGER PRIMARY KEY,
        title TEXT,
        author TEXT
    )
''')
n_users = 30
# Tirer 10 livres communs une seule fois en dehors de la boucle pour un resultat plus logique
common_indices = np.random.choice(len(books), 10, replace=False)
common_book_ids = [int(books.iloc[idx]['id']) for idx in common_indices]
for user_id in range(n_users):
    # 10 livres communs à tous les users pour harmoniser les donnees
    # 10 livres aléatoires propres à chaque user pour plus de realisme
    random_indices = np.random.choice(len(books), 10, replace=False)
    random_book_ids = [int(books.iloc[idx]['id']) for idx in random_indices]
    user_books = common_book_ids + random_book_ids
    for book_id in user_books:
        note = np.random.choice([1.0, 0.6, 0.4,0.2,-1.0])
        cursor.execute(
            'INSERT OR IGNORE INTO interactions (user_id, book_id, note) VALUES (?, ?, ?)',
            (user_id, book_id, note)
        )
conn.commit() # Sauvegarder toutes les modifications dans la base de données.

# Remplir la table 'books_rated' avec les détails des livres qui ont reçu des notes.
# Charger toutes les interactions dans un DataFrame pour extraire les IDs de livres uniques.
df_interactions = pd.read_sql('SELECT * FROM interactions', conn)
rated_book_ids = df_interactions['book_id'].unique() # Obtenir la liste unique des IDs de livres notés.
# Pour chaque ID de livre noté, récupérer ses détails (titre, auteur) et l'insérer dans 'books_rated'.
for book_id in rated_book_ids:
    book = books[books['id'] == book_id] # Trouver le livre correspondant dans le DataFrame 'books'.
    if not book.empty:
        # Insérer ou ignorer si le livre existe déjà dans 'books_rated'.
        cursor.execute(
            'INSERT OR IGNORE INTO books_rated (book_id, title, author) VALUES (?, ?, ?)',
            (int(book_id), str(book.iloc[0]['title']), str(book.iloc[0]['author']))
        )
conn.commit()
print(f"\n {len(df_interactions)} interactions stockées")
print(f" {len(rated_book_ids)} livres notés")


 7428 interactions stockées
 2908 livres notés


# Recommandation Hybride — User-Based + Content-Based

## Principe

On combine deux approches complémentaires pour pallier les limites de chacune :

- **Content-Based** — recommande des livres similaires à ceux que l'utilisateur a aimés, basé sur le contenu (description, genre, auteur)
- **User-Based** — recommande des livres aimés par des utilisateurs ayant les mêmes goûts

---

## Fonctionnement

```
Utilisateur
     │
     ├── Interactions (likes, notes)
     │         │
     │         ├── Content-Based → livres similaires au contenu
     │         │
     │         └── User-Based → livres aimés par users similaires
     │
     └── Résultat combiné → liste finale de recommandations
```

---

## Pourquoi combiner les deux ?

| Situation | Content-Based | User-Based |
|---|---|---|
| Nouveau livre sans interactions | ✅ fonctionne | ❌ invisible |
| Utilisateur avec peu d'interactions | ✅ fonctionne | ❌ similarité faible |
| Découverte hors du contenu habituel | ❌ limité | ✅ fonctionne |
| Livres populaires dans une communauté | ❌ limité | ✅ fonctionne |

La combinaison des deux couvre les angles morts de chaque approche.

---

## Limites

- Les données simulées réduisent la pertinence du User-Based
- Le cold start reste un problème pour les tout nouveaux utilisateurs
- La qualité des recommandations dépend directement du volume d'interactions

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import sqlite3
conn=sqlite3.connect('interactions.db')
#on va construire la matrice qui contient les notes de chaque utilisateur pour les livres
matrice=pd.read_sql("select * from interactions",conn)
#on contruit la matrice des actions de chaque utilisateur
matrice=matrice.pivot_table(
    index='user_id',
    columns='book_id',
    values='note',
    fill_value=0
)
#on enregistre la matrice pour un usage futur sans recalculer
np.save('matrice_user_collab.npy',matrice)
#on va calculer la similarite cosinus entre les utilisateurs
print(f"La shape de la matrice principale est {matrice.shape}")
#on va creer une fonction test pour recommander des livres a un utilisateur
#on cree une fonction qui va recuperer toutes les notes des livres notes par les utilisateurs similaires et pas encore note par cet utilisateur
def get_books_of_similar(users, userid):
    cursor = conn.cursor()
    ids = [user[0] for user in users]
    # On recupere les infos des livres notes par les users similaires a lui mais qu'il na pas encore note
    placeholders = ','.join(['?' for _ in ids])
    query = f"""
        SELECT i.book_id, i.user_id, i.note, b.title
        FROM interactions AS i
        JOIN books_rated AS b ON i.book_id = b.book_id
        WHERE i.user_id IN ({placeholders})
        AND i.book_id NOT IN (
            SELECT book_id FROM interactions WHERE user_id = ?
        )
        ORDER BY i.book_id
    """
    params = tuple(ids) + (userid,)
    cursor.execute(query, params)
    df = pd.DataFrame(cursor.fetchall(), columns=['book_id', 'user_id', 'note', 'title'])
    # Garder les livres notés par au moins 2 users similaires
    df = df.groupby('book_id').filter(lambda x: len(x)>=2)
    return df
#on creer une fonction qui recoit les infos des livres et des users et retourne les id des livres les plus aptes a etre recommandes
def choisir_livre(infos_livres):
  #on va calculer la note ponderee de chaque note par rapport a la  similarite
  infos_livres['weighted_rating']=infos_livres['note']*infos_livres['similarity']
  #on va calcucler la note moyenne de chaque livre
  infos_livres=infos_livres.groupby('book_id').agg(
      weighted_rating_sum=('weighted_rating','sum'),
      similarity_sum=('similarity','sum'),
      title=('title','first')
  )
  infos_livres['score']=infos_livres['weighted_rating_sum']/infos_livres['similarity_sum']
  #on supprime les colonnes non utiles
  infos_livres.drop(columns=['weighted_rating_sum','similarity_sum'],inplace=True)
  #on trie la liste par ordre decroissant
  infos_livres=infos_livres.sort_values('score',ascending=False)
  #on choisit les livres avec au moins 3 etoiles
  infos_livres=infos_livres[infos_livres['score']>=0.5]
  # Reset index to make 'book_id' a column
  infos_livres = infos_livres.reset_index()
  return infos_livres
def recommander_to_user(id):
  #on calcule d'abord le cosine similarity de ce user avec les autres
  similarity=cosine_similarity(matrice.loc[[id]],matrice).flatten()
  similarity=sorted(list(enumerate(similarity)),key=lambda x:x[1],reverse=True)[1:11]
  books_of_similar=get_books_of_similar(similarity,id)
  #on va assembler les deux resultats pour pouvoir faciliter les calculs
  books_of_similar=books_of_similar.merge(pd.DataFrame(similarity,columns=['user_id','similarity']),on='user_id')
  resultat=choisir_livre(books_of_similar)
  return (resultat,resultat['title'])
#on va maintenant combiner l'approche content based et user based pour un meilleur resulat
def recommander_user_content_based(id):
  user_based=recommander_to_user(id)[0]
  interactions=pd.read_sql(f"select * from interactions where user_id={id}",conn)
  #on creer un set avec les couples book_id:note
  interactions=dict(zip(interactions['book_id'],interactions['note']))
  content_based=recommand_interactions_word2vec(interactions,10)
  #on regroupe les deux et on les affiches
  overall_result=pd.concat([user_based,content_based])
  overall_result=overall_result.drop_duplicates(subset='book_id')
  overall_result=overall_result.drop(columns=['score'])
  return overall_result
print(recommander_user_content_based(6))

La shape de la matrice principale est (30, 2908)
Here are the books you might like:
     book_id                                              title
0    1307394                   Tenggelamnya Kapal Van Der Wijck
1   13112023                                    One Breath Away
2   11767984          My Father's Parrot (Back to the Land, #1)
3    1727464                                   Playing the Jack
4     199635                             Fair and Tender Ladies
5    6814059               Le Club des incorrigibles optimistes
6    6869581  ÃÂÃÂ ÃÂ£ÃÂÃÂª ÃÂ£ÃÂÃÂÃÂ§ ÃÂ§ÃÂ...
7      13979                                              Aerie
8      73966                    When God Writes Your Love Story
9     152838                                These Strange Ashes
10   1304525  The Language Police:  How Pressure Groups Rest...
11     29488                                              Candy
12  24040192  The New Tsar: The Rise and Reign of Vladimir P...
13   7097866        